In [ ]:
import numpy as np
from PIL import Image

image_path = (
    "data/shadow-gen/stepho_renderings/reflections_4/train/mercedes-reflection_0069.jpg"
)
image = Image.open(image_path)

im_arr = np.array(image)
image.resize((512, 512))

In [ ]:
# cut = Image.open("data/shadow-gen/stepho_renderings/reflections_4/train/audi-reflection_0001_cut.jpg")
cut_path = image_path.replace(".jpg", "_cut.jpg")

cut = Image.open(cut_path)
cut.resize((512, 512))

In [ ]:
bg_color = im_arr[10, 10]
bg_sample = np.zeros((64, 64, 3))
bg_sample[:] = bg_color
Image.fromarray(bg_sample.astype(np.uint8))

In [ ]:
(np.array(cut) != 255).sum() / (cut.size[0] * cut.size[1]) * 100

In [ ]:
import cv2
import numpy as np

# cut_path = image_path.replace(".jpg", "_cut.jpg")
# cut = Image.open(cut_path)
# bg = np.array(image) - np.array(cut)
# Image.fromarray(bg)
# bg
car_mask = (np.array(cut) < 255).astype(np.uint8)[:, :, 0]
car_mask = cv2.dilate(car_mask, np.ones((3, 3), np.uint8), iterations=2)
car_mask = cv2.erode(car_mask, np.ones((3, 3), np.uint8), iterations=3)
Image.fromarray(car_mask * 255, mode="L").resize((512, 512))
# (np.array(cut) < 255).astype(np.uint8)*255

In [ ]:
masked_im = im_arr.copy()
masked_im[car_mask != 0] = bg_color
Image.fromarray(masked_im.astype(np.uint8)).resize((512, 512))

In [ ]:
np.percentile(im_arr, 50, axis=(0, 1))

In [ ]:
refl_mask = (masked_im < 215).astype(np.uint8)[
    :, :, 0
]  # TODO Replace 215 with a value from the images stats
Image.fromarray((refl_mask * 255), mode="L").resize((512, 512))

In [ ]:
opened_mask = cv2.morphologyEx(
    refl_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=8
)
Image.fromarray((opened_mask * 255).astype(np.uint8), mode="L").resize((512, 512))

In [ ]:
closed_mask = cv2.morphologyEx(
    opened_mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=12
)
Image.fromarray((closed_mask * 255).astype(np.uint8), mode="L").resize((512, 512))

In [ ]:
blurred_mask = cv2.GaussianBlur(opened_mask, (5, 5), 0)
Image.fromarray((opened_mask * 255).astype(np.uint8), mode="L").resize((512, 512))